In [1]:
# Install MPI on Colab
!apt-get update -qq
!apt-get install -y -qq mpich


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libslurm37.
(Reading database ... 126675 files and directories currently installed.)
Preparing to unpack .../libslurm37_21.08.5-2ubuntu1_amd64.deb ...
Unpacking libslurm37 (21.08.5-2ubuntu1) ...
Selecting previously unselected package hwloc-nox.
Preparing to unpack .../hwloc-nox_2.7.0-2ubuntu1_amd64.deb ...
Unpacking hwloc-nox (2.7.0-2ubuntu1) ...
Selecting previously unselected package libmpich12:amd64.
Preparing to unpack .../libmpich12_4.0-3_amd64.deb ...
Unpacking libmpich12:amd64 (4.0-3) ...
Selecting previously unselected package mpich.
Preparing to unpack .../archives/mpich_4.0-3_amd64.deb ...
Unpacking mpich (4.0-3) ...
Selecting previously unselected package libmpich-dev:amd64.
Preparing to unpack .../libmpich-dev_4.0-3_amd64.deb ...
Unpacking libmpich

In [4]:
%%writefile mpi_vector_sum.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char* argv[]) {
    int rank, size;
    long N = 10000000; // total elements

    double *A = NULL;
    double *local_A = NULL;
    double local_sum = 0.0, global_sum = 0.0, global_avg = 0.0;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    long base = N / size;
    long remainder = N % size;
    long local_n = base + (rank < remainder ? 1 : 0);

    // ---- use int for MPI arrays ----
    int *counts = (int*)malloc(size * sizeof(int));
    int *displs = (int*)malloc(size * sizeof(int));

    for (int i = 0; i < size; i++) {
        counts[i] = (int)(base + (i < remainder ? 1 : 0));
        displs[i] = (i == 0) ? 0 : displs[i - 1] + counts[i - 1];
    }

    // Allocate and initialize data on root
    if (rank == 0) {
        A = (double*)malloc(N * sizeof(double));
        for (long i = 0; i < N; i++)
            A[i] = i + 1;
    }

    local_A = (double*)malloc(local_n * sizeof(double));

    double start_time = MPI_Wtime();

    // Scatterv for uneven chunks
    MPI_Scatterv(A, counts, displs, MPI_DOUBLE, local_A, local_n, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    // Local computation
    for (long i = 0; i < local_n; i++)
        local_sum += local_A[i];

    // Combine results using Allreduce (sum + average)
    MPI_Allreduce(&local_sum, &global_sum, 1, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);
    global_avg = global_sum / N;

    double end_time = MPI_Wtime();
    double elapsed = end_time - start_time;

    if (rank == 0) {
        double expected = (N * (N + 1)) / 2.0;
        printf("\n=== MPI Parallel Vector Sum ===\n");
        printf("Processes        : %d\n", size);
        printf("Total Elements   : %ld\n", N);
        printf("Computed Sum     : %.0f\n", global_sum);
        printf("Expected Sum     : %.0f\n", expected);
        printf("Difference       : %.5f\n", expected - global_sum);
        printf("Computed Average : %.5f\n", global_avg);
        printf("Elapsed Time     : %.6f sec\n", elapsed);
    }

    free(local_A);
    free(counts);
    free(displs);
    if (rank == 0) free(A);

    MPI_Finalize();
    return 0;
}


Overwriting mpi_vector_sum.c


In [8]:
!mpicc mpi_vector_sum.c -o mpi_vector_sum


In [7]:
!mpirun --allow-run-as-root --oversubscribe -np 4 ./mpi_vector_sum



=== MPI Parallel Vector Sum ===
Processes        : 4
Total Elements   : 10000000
Computed Sum     : 50000005000000
Expected Sum     : 50000005000000
Difference       : 0.00000
Computed Average : 5000000.50000
Elapsed Time     : 0.111233 sec
